# soulclip on Colab — a scene script into a stitched AI film

Uses **Wan 2.1 T2V 1.3B** plus the **CausVid step-distilled LoRA** on Colab's
free T4. No API key, no payment.

**Before you start:** `Runtime > Change runtime type > T4 GPU`.

### Why `--fast` helps

The default runs 20 denoising steps *with* classifier-free guidance, and CFG
runs the model twice per step — 40 forward passes per clip. CausVid is
distilled for 4 steps at guidance 1.0, so CFG disappears: **4 passes instead
of 40, a 10x reduction in work.**

That ratio is arithmetic and is reliable. **What one pass costs in seconds on
a T4 is not something this notebook can tell you in advance** — published
figures for this GPU class span 2-10 minutes per clip, a 5x spread.

So rather than quote a number: **step 6 measures itself.** After each clip it
prints the running average and a projected finish:

```
[3/60] scene 2 (part 1/6): saved scene_003.mp4 (41.2s)
      avg 42s/clip · 57 left · ~40 min to go
```

Run 5-6 clips, read that line, and you will know your real total. Rough
bracket while you wait: **35 min to 2 h** for 60 clips with `--fast`.

### Colab free-tier limits

| | |
|---|---|
| Max session | 12 hours while actively computing |
| Idle timeout | ~90 min (only when nothing is running) |
| Weekly GPU quota | ~15-30 hours |

**Quality note:** 4-step output is softer than 20-step. Try 6 clips each way.


## 1. Check the GPU


In [ ]:
!nvidia-smi

import torch
assert torch.cuda.is_available(), (
    'No GPU. Runtime > Change runtime type > T4 GPU, then rerun.')
p = torch.cuda.get_device_properties(0)
print(f'{p.name}, {p.total_memory/1e9:.1f} GB VRAM')


## 2. Install

Takes 2-3 minutes.


In [ ]:
!pip install -q -U diffusers transformers accelerate ftfy imageio imageio-ffmpeg
!git clone -q https://github.com/Naserkhan07/soul_exter.git 2>/dev/null || true
%cd /content/soul_exter
!git checkout -q arena/019f98a2-soul-exter && git pull -q
print('ready')


## 3. Keep your work across disconnects (recommended)

Saves clips to Drive so a dropped session costs you nothing. Skip this
cell if you would rather not connect Drive — but then a disconnect loses
everything generated so far.


In [ ]:
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORKDIR = '/content/drive/MyDrive/soulclip/work'
    OUTPUT  = '/content/drive/MyDrive/soulclip/film.mp4'
else:
    WORKDIR = '/content/work'
    OUTPUT  = '/content/film.mp4'

print('clips ->', WORKDIR)
print('film  ->', OUTPUT)


## 4. Your script

Label scenes `Scene 1:`, `Scene 2:` ... or separate them with blank lines.

**Prompt tips for Wan:** describe the *camera* and the *motion*, not just
the subject — 'slow dolly in', 'waves crash', 'hair moves in the wind'.
Repeat character details in every scene; the model has no memory between
clips.


In [ ]:
script = '''
Scene 1: A lighthouse on a black rock headland at dusk, its beam sweeping
slowly across heavy grey water. Rain streaks sideways. Slow dolly in.

Scene 2: Inside the lantern room, brass fittings glowing warm. An old
keeper in a wool coat winds a mechanism by hand. Firelight flickers.

Scene 3: Waves crash white over a dark reef, spray flung high into the
storm. Handheld camera, violent motion.

Scene 4: A small fishing boat pinned against the rocks, mast broken, a
single lantern swinging wildly on the deck.

Scene 5: The keeper hauls a heavy lever with both hands, straining. The
great beam swings and holds steady.

Scene 6: Dawn over a calm flat sea, pale gold light. Two figures wrapped
in blankets sit on stone steps, steam rising from tin mugs.
'''

with open('/content/script.txt', 'w') as f:
    f.write(script)

!python -m soulclip.cli scenes /content/script.txt --clip-seconds 5


## 5. Settings

`CLIPS` is the main dial. Each clip is ~5 s, so 6 clips = 30 s of film.

Leave `STEPS` at 20 for good quality, or drop to 10 to roughly halve the
time at some cost in sharpness.


In [ ]:
CLIPS = 6            # START HERE. Read the ETA, then raise to 60.
FAST  = True         # CausVid LoRA: 4 steps, no CFG (10x less work)
WIDTH, HEIGHT = 832, 480   # 640x368 is ~2x cheaper again

STYLE = 'cinematic anime, detailed background art, dramatic lighting, film grain'

print(f'{CLIPS} clips x 5.06s = {CLIPS*5.06:.0f}s of film')
print('Per-clip speed is measured during the run and printed as an ETA.')
print('A full 5-minute film is 60 clips.')


## 6. Generate

The first run downloads ~6 GB of weights (a few minutes, once per session).

**If it disconnects, just run this cell again** — finished clips are reused
and only the missing ones are generated.


In [ ]:
FAST_FLAG = '--fast' if FAST else ''

!python -m soulclip.cli render /content/script.txt \
    --provider wan $FAST_FLAG \
    --clip-seconds 5 \
    --max-scenes $CLIPS \
    --target $((CLIPS*5)) \
    --width $WIDTH --height $HEIGHT \
    --style "$STYLE" \
    --workdir "$WORKDIR" \
    -o "$OUTPUT"


## 7. Watch it


In [ ]:
from IPython.display import HTML
from base64 import b64encode

data = b64encode(open(OUTPUT, 'rb').read()).decode()
HTML(f'<video width=640 controls src="data:video/mp4;base64,{data}"></video>')


## 8. Download


In [ ]:
from google.colab import files
files.download(OUTPUT)


---
## Working out your real total

During step 6 each clip prints its own time plus a projection:

```
      avg 42s/clip · 57 left · ~40 min to go
```

Multiply your measured `avg` by 60 for the full film:

| Your avg | 60 clips |
|---|---|
| 20 s | ~20 min |
| 40 s | ~40 min |
| 60 s | ~1 hour |
| 120 s | ~2 hours |

Add ~8 min setup and ~1 min stitching (stitching was measured directly: 60
clips join in about a minute).

### Clip counts (measured)

One clip is 81 frames @ 16fps = **5.06 s**.

| | Clips | Final length |
|---|---|---|
| Hard cuts | **60** | 5m05s |
| 0.4 s crossfades | **65** | 5m03s |

Crossfades overlap clips (64 x 0.4 s ≈ 26 s lost), so they need 5 more.

### If your measured rate is too slow

- `640x368` instead of `832x480` — roughly halves it
- `--wan-frames 49` — 3 s clips, ~40% cheaper each
- Two notebooks on different scene ranges (Colab allows 2 sessions)

### Honest expectations

Wan 1.3B at 4 steps is the fastest usable setup, not the best-looking. Output
is 480p and well short of paid Kling or Veo. Characters will not stay
consistent between shots — that is the model, not the pipeline.
